# Multi-Model Fair-RAG Notebook (Existing Results + T5-Base + SPLADE)



This notebook does three things:

1. loads and summarizes experiment outputs already produced in this repo (currently flanT5Small + BM25)

2. provides a reusable run scaffold for the same pipeline with flanT5Base

3. prepares a SPLADE retrieval path for later runs with the same evaluation flow



Design goals:

- keep existing results visible and comparable

- avoid accidental destructive reruns

- keep model/retriever changes confined to config cells

In [1]:
from pathlib import Path
from datetime import datetime
import importlib
import subprocess
import time

import pandas as pd
from IPython.display import display

import notebook_experiment_utils as neu
importlib.reload(neu)

ExperimentConfig = neu.ExperimentConfig
find_repo_root = neu.find_repo_root
resolve_python_executable = neu.resolve_python_executable
ensure_paths_exist = neu.ensure_paths_exist

run_experiment_for_alpha = neu.run_experiment_for_alpha
run_retriever_grid = neu.run_retriever_grid
run_deterministic_reference = neu.run_deterministic_reference
run_mmr_deterministic = neu.run_mmr_deterministic
run_gold_deterministic_reference = neu.run_gold_deterministic_reference
run_gold_baseline = neu.run_gold_baseline

normalize_eu_grid = neu.normalize_eu_grid
normalize_eu_for_retriever = neu.normalize_eu_for_retriever
load_normalized_rows = neu.load_normalized_rows
load_raw_rows = neu.load_raw_rows

raw_results_path = neu.raw_results_path
add_ee_d_interval_bins = neu.add_ee_d_interval_bins
summarize_by_interval = neu.summarize_by_interval

In [ ]:
ROOT = find_repo_root()

PY = resolve_python_executable(ROOT)



profiles = {

    "weak": {"max_queries": 30, "n_samples": 10, "k": 5, "print_interval": 10},

    "balanced": {"max_queries": None, "n_samples": 10, "k": 5, "print_interval": 40},

    "strong": {"max_queries": None, "n_samples": 100, "k": 5, "print_interval": 100},

}



# Switch to weak profile for initial T5-Base testing

profile_name = "balanced"

profile = profiles[profile_name]



existing_generator = "flanT5Small"

target_generator = "flanT5Base"



cfg_existing = ExperimentConfig(

    root=ROOT,

    python_exe=PY,

    generator_name=existing_generator,

    lamp_num=4,

    retriever_name="bm25",

    alphas=(1, 2, 4, 8),

    max_queries=profile["max_queries"],

    n_samples=profile["n_samples"],

    k=profile["k"],

    remove_temp_files=True,

    skip_existing=True,

    mmr_base_retriever="bm25",

    mmr_lambda=0.6,

    run_tag=f"{profile_name}_{existing_generator}",

    print_interval=profile["print_interval"],

)



cfg_t5base_bm25 = ExperimentConfig(

    root=ROOT,

    python_exe=PY,

    generator_name=target_generator,

    lamp_num=4,

    retriever_name="bm25",

    alphas=(1, 2, 4, 8),

    max_queries=profile["max_queries"],

    n_samples=profile["n_samples"],

    k=profile["k"],

    remove_temp_files=True,

    skip_existing=True,

    mmr_base_retriever="bm25",

    mmr_lambda=0.6,

    run_tag=f"{profile_name}_{target_generator}_bm25",

    print_interval=profile["print_interval"],

)



cfg_t5base_splade = ExperimentConfig(

    root=ROOT,

    python_exe=PY,

    generator_name=target_generator,

    lamp_num=4,

    retriever_name="splade",

    alphas=(1, 2, 4, 8),

    max_queries=profile["max_queries"],

    n_samples=profile["n_samples"],

    k=profile["k"],

    remove_temp_files=True,

    skip_existing=True,

    mmr_base_retriever="splade",

    mmr_lambda=0.6,

    run_tag=f"{profile_name}_{target_generator}_splade",

    print_interval=profile["print_interval"],

)



ensure_paths_exist([cfg_existing.python_exe])



print("Root      :", ROOT)

print("Python    :", cfg_existing.python_exe)

print("Profile   :", profile_name)

print("Existing  :", cfg_existing.generator_name, cfg_existing.retriever_name)

print("Target A  :", cfg_t5base_bm25.generator_name, cfg_t5base_bm25.retriever_name)

print("Target B  :", cfg_t5base_splade.generator_name, cfg_t5base_splade.retriever_name)

Root      : /Users/asimk/Code/Fair-RAG
Python    : /Users/asimk/Code/Fair-RAG/.venv/bin/python
Profile   : weak
Existing  : flanT5Small bm25
Target A  : flanT5Base bm25
Target B  : flanT5Base splade


## Load Existing Results Already Produced Here

This reads from `experiment_results/flanT5Small/lamp4/bm25/notebook_outputs_*` and displays key tables without rerunning experiments.

In [3]:
def _latest_notebook_output_dirs(cfg: ExperimentConfig) -> list[Path]:
    base = cfg.root / "experiment_results" / cfg.generator_name / f"lamp{cfg.lamp_num}" / cfg.retriever_name
    if not base.exists():
        return []
    out_dirs = [p for p in base.glob("notebook_outputs_*") if p.is_dir()]
    out_dirs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return out_dirs

def _safe_read_csv(fp: Path) -> pd.DataFrame | None:
    if fp.exists():
        return pd.read_csv(fp)
    return None

def show_existing_outputs(cfg: ExperimentConfig, max_dirs: int = 2) -> None:
    dirs = _latest_notebook_output_dirs(cfg)[:max_dirs]
    if not dirs:
        print("No notebook_outputs_* directories found for", cfg.generator_name, cfg.retriever_name)
        return

    for out_dir in dirs:
        print(f"\n=== Existing outputs: {out_dir.name} ===")

        preferred_files = [
            "ee_d_interval_summary.csv",
            "pvalues_vs_deterministic_by_interval.csv",
            "tableA_normalized_eu_by_disparity_interval_pooled.csv",
            "tableB_eu_diff_st_minus_det_by_disparity_interval_pooled.csv",
            "tableA_raw_eu_by_disparity_interval_pooled.csv",
            "tableB_raw_eu_diff_st_minus_det_by_disparity_interval_pooled.csv",
        ]

        for name in preferred_files:
            fp = out_dir / name
            df = _safe_read_csv(fp)
            if df is not None:
                print(f"\n{name} (rows={len(df)}):")
                display(df.head(20))

        perq_fp = out_dir / "per_query_all_metrics.csv"
        perq = _safe_read_csv(perq_fp)
        if perq is not None:
            print("\nPer-query snapshot:")
            display(perq.head(10))
            print("Per-query rows:", len(perq), "| unique qids:", perq["qid"].nunique())

In [4]:
show_existing_outputs(cfg_existing, max_dirs=2)

# Quick high-level summaries from existing raw deterministic files, if present.
det_fp = raw_results_path(cfg_existing, cfg_existing.retriever_name, 1, output_suffix="_deterministic")
mmr_fp = raw_results_path(cfg_existing, "mmr", 1, output_suffix="_mmr_deterministic")

rows = []
if det_fp.exists():
    df = load_raw_rows(cfg_existing, cfg_existing.retriever_name, (1,), output_suffix="_deterministic")
    rows.append({
        "run": f"{cfg_existing.retriever_name}_deterministic",
        "n_qids": int(df["qid"].nunique()),
        "mean_ee_d": float(df["ee_d"].mean()),
        "mean_ee_r": float(df["ee_r"].mean()),
        "mean_eu": float(df["eu"].mean()),
    })
if mmr_fp.exists():
    df = load_raw_rows(cfg_existing, "mmr", (1,), output_suffix="_mmr_deterministic")
    rows.append({
        "run": "mmr_deterministic",
        "n_qids": int(df["qid"].nunique()),
        "mean_ee_d": float(df["ee_d"].mean()),
        "mean_ee_r": float(df["ee_r"].mean()),
        "mean_eu": float(df["eu"].mean()),
    })

if rows:
    print("\nExisting deterministic summaries:")
    display(pd.DataFrame(rows))


=== Existing outputs: notebook_outputs_weak ===

ee_d_interval_summary.csv (rows=5):


,ee_d_interval,n_queries,mean_eu,mean_ee_d,mean_ee_r
0,0.0-0.2,28,0.174400,0.148163,0.167585
1,0.2-0.4,10,0.198856,0.270182,0.180899
2,0.4-0.6,6,0.135011,0.505714,0.298298
3,0.6-0.8,15,0.195331,0.698133,0.214723
4,0.8-1.0,30,0.140095,0.948316,0.172128



pvalues_vs_deterministic_by_interval.csv (rows=2):


,ee_d_interval,metric,n_qids,p_two_sided
0,0.8-1.0,ee_r,30,0.317311
1,0.8-1.0,eu,30,0.248864



Per-query snapshot:


...

,ee_d_interval,n_qids,mean_eu_stochastic_raw,mean_eu_deterministic_raw,eu_diff_st_minus_det_raw,mean_ee_d_stochastic,mean_ee_r_stochastic
0,[0.0-0.2),581,0.032601,0.042184,-0.009584,0.121214,0.209077
1,[0.2-0.4),259,0.051992,0.055496,-0.003504,0.298654,0.337426
2,[0.4-0.6),275,0.049057,0.053525,-0.004468,0.499675,0.370179
3,[0.6-0.8),452,0.047974,0.047033,0.000941,0.711835,0.319770
4,[0.8-1.0],804,0.047702,0.047482,0.000220,0.950623,0.313180



Per-query snapshot:


,retriever,alpha,qid,ee_d,ee_r,eu,ee_d_interval
0,bm25,1,312,0.375000,0.533333,0.265274,0.2-0.4
1,bm25,1,313,0.433333,0.583333,0.254094,0.4-0.6
2,bm25,1,315,0.141667,0.040071,0.143986,0.0-0.2
3,bm25,1,316,0.133333,0.166667,0.286112,0.0-0.2
4,bm25,1,317,0.113889,0.100000,0.131660,0.0-0.2
5,bm25,1,318,0.155556,0.116667,0.139731,0.0-0.2
6,bm25,1,319,0.147222,0.000000,0.074386,0.0-0.2
7,bm25,1,3110,0.130556,0.166667,0.202954,0.0-0.2
8,bm25,1,3111,0.125000,0.103933,0.322173,0.0-0.2
9,bm25,1,3112,0.138889,0.133333,0.049870,0.0-0.2


Per-query rows: 3332 | unique qids: 833

Existing deterministic summaries:


,run,n_qids,mean_ee_d,mean_ee_r,mean_eu
0,bm25_deterministic,833,1.0,0.315762,0.048012
1,mmr_deterministic,833,1.0,0.310872,0.049176


## Reusable Pipeline Runner (Model/Retriever Agnostic)



Toggle flags and select a config (`cfg_t5base_bm25` or `cfg_t5base_splade`) to run the same stages used in your current notebook.

In [5]:
RUN_GOLD = True
RUN_BM25_OR_OTHER_GRID = False
RUN_DETERMINISTIC_REF = True
RUN_MMR_DETERMINISTIC = False
RUN_GOLD_DETERMINISTIC = False
RUN_NORMALIZE = False
RUN_INTERVAL_SUMMARY = False

DETERMINISTIC_ALPHA = 1

def run_stage(name: str, enabled: bool, fn):
    if not enabled:
        print(f"[skip] {name}")
        return
    print(f"\n=== START {name} @ {datetime.now().strftime('%H:%M:%S')} ===")
    t0 = time.time()
    fn()
    print(f"=== DONE {name} in {(time.time()-t0)/60:.2f} min ===")

def retrieval_results_fp(cfg: ExperimentConfig, retriever_name: str) -> Path:
    return cfg.root / "retrieval" / "retrieval_results" / cfg.generator_name / retriever_name / f"{cfg.lamp_num}.json"

def build_splade_retrieval(cfg: ExperimentConfig, checkpoint: str = "naver/splade_v2_max") -> None:
    fp = retrieval_results_fp(cfg, "splade")
    if fp.exists():
        print("SPLADE retrieval already exists:", fp)
        return

    cmd = [
        str(cfg.python_exe),
        "retrieval/rank_profiles.py",
        "--lamp_num", str(cfg.lamp_num),
        "--ranker", "splade",
        "--generator_name", cfg.generator_name,
        "--splade_checkpoint", checkpoint,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=cfg.root, check=True)

    if not fp.exists():
        raise FileNotFoundError(f"SPLADE retrieval build finished but file not found: {fp}")

def run_core_pipeline(cfg: ExperimentConfig) -> None:
    print("\nRunning config:", cfg)

    run_stage("gold baseline", RUN_GOLD, lambda: run_gold_baseline(cfg, alpha=8))
    run_stage(
        "deterministic reference",
        RUN_DETERMINISTIC_REF,
        lambda: run_deterministic_reference(
            cfg, alpha=DETERMINISTIC_ALPHA, output_suffix="_deterministic"
        ),
    )
    run_stage(
        "deterministic mmr",
        RUN_MMR_DETERMINISTIC,
        lambda: run_mmr_deterministic(
            cfg, alpha=DETERMINISTIC_ALPHA, output_suffix="_mmr_deterministic"
        ),
    )
    run_stage(
        "gold deterministic",
        RUN_GOLD_DETERMINISTIC,
        lambda: run_gold_deterministic_reference(
            cfg, alpha=DETERMINISTIC_ALPHA, output_suffix="_gold_deterministic"
        ),
    )

    run_stage("retriever alpha grid", RUN_BM25_OR_OTHER_GRID, lambda: run_retriever_grid(cfg))
    run_stage(
        "normalize alpha grid",
        RUN_NORMALIZE,
        lambda: normalize_eu_grid(cfg, normalization_scope="all-settings"),
    )

    if RUN_INTERVAL_SUMMARY:
        def _interval():
            df = load_normalized_rows(cfg)
            df = add_ee_d_interval_bins(df)
            out = summarize_by_interval(df)
            print("Interval summary:")
            display(out)

        run_stage("normalized interval summary", True, _interval)

## Run Targets



- Target A: flanT5Base + BM25

- Target B: flanT5Base + SPLADE



Typical order:

1. run Target A first to isolate generator effects

2. then build SPLADE retrieval and run Target B for retriever swap analysis

In [ ]:
RUN_TARGET_T5BASE_BM25 = True

RUN_TARGET_T5BASE_SPLADE = False



if RUN_TARGET_T5BASE_BM25:

    run_core_pipeline(cfg_t5base_bm25)



if RUN_TARGET_T5BASE_SPLADE:

    # One-time retrieval generation for SPLADE (if missing).

    build_splade_retrieval(cfg_t5base_splade)

    run_core_pipeline(cfg_t5base_splade)


Running config: ExperimentConfig(root=PosixPath('/Users/asimk/Code/Fair-RAG'), python_exe=PosixPath('/Users/asimk/Code/Fair-RAG/.venv/bin/python'), generator_name='flanT5Base', lamp_num=4, retriever_name='bm25', alphas=(1, 2, 4, 8), max_queries=30, n_samples=10, k=5, remove_temp_files=True, skip_existing=True, mmr_base_retriever='bm25', mmr_lambda=0.6, run_tag='weak_flanT5Base_bm25', print_interval=10)

=== START gold baseline @ 12:12:05 ===
[prep] missing retrieval file: /Users/asimk/Code/Fair-RAG/retrieval/retrieval_results/flanT5Base/gold/4.json

> /Users/asimk/Code/Fair-RAG/.venv/bin/python retrieval/gold_retriever.py --lamp_num 4 --generator_name flanT5Base

> /Users/asimk/Code/Fair-RAG/.venv/bin/python experiment.py --generator_name flanT5Base --lamp_num 4 --retriever_name gold --alpha 8 --k 5 --n_samples 10 --max_queries 30 --remove_temp_files --run_tag weak_flanT5Base_bm25 --print_interval 10
[run-log] /Users/asimk/Code/Fair-RAG/experiment_results/runs/20260403_121208_gold_alp

## SPLADE vs Contriever (Practical Note)

Short answer: SPLADE is often lighter at query time than Contriever for large corpora because retrieval is sparse/lexical-style over an index, while Contriever relies on dense embeddings and vector search.

Caveats:
- SPLADE indexing can be heavy upfront and index storage may grow
- Contriever can be very fast if you already maintain optimized ANN infrastructure
- end-to-end cost depends on corpus size, hardware, and whether embeddings are precomputed